# Eval checkpoint — metric CHÍNH THỨC (đúng protocol paper)

Load weight đã train (trên Drive) → chạy `test()` của code → ra **Recall/HR/NDCG** đúng cách đo của paper.
**KHÔNG train lại, chạy CPU được** (eval nhẹ). Dùng số này để đối chiếu paper (KHÔNG dùng số bảng demo).

## 1. Setup

In [12]:
!pip install -q absl-py setproctitle deprecated faiss-cpu
import os, glob, re, torch
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'
!git log --oneline -1

/content/DIVRS
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1805a42 (HEAD -> main, origin/main, origin/HEAD) Demo cell A: use code's metrics.Metrics + eval all users (matches paper); pretty pandas table


## 2. Tìm checkpoint DIVRS (epoch cao nhất trên Drive)

In [13]:
runs=[r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' in r.lower() and glob.glob(r+'ckpt/epoch_*.pth')]
run=max(runs,key=lambda r:len(glob.glob(r+'ckpt/epoch_*.pth')))
cks=glob.glob(run+'ckpt/epoch_*.pth')
CKPT=max(cks,key=lambda x:int(re.findall(r'epoch_(\d+)',x)[0]))
print('Run:', os.path.basename(run.rstrip('/')))
print('So checkpoint:', len(cks), '| dung epoch cao nhat:', os.path.basename(CKPT))

Run: ml10m-DIVRS-0.8-cause-0.2-Res-debug_2026-06-16-16-32-50
So checkpoint: 35 | dung epoch cao nhat: epoch_34.pth


## 3. Chạy TEST chính thức trên checkpoint đó
Tạo output mới `eval_out/`. Eval nhẹ (chỉ rank embedding), CPU vài phút.

In [ ]:
USE_GPU=torch.cuda.is_available(); print('use_gpu =', USE_GPU)
!python app.py \
  --flagfile config/ml10m_DIVRS.cfg \
  --test=True --test_ckpt="{CKPT}" \
  --output ./eval_out/ \
  --use_gpu={USE_GPU} --gpu_id=0 \
  --cg_use_gpu=False --num_workers=2

use_gpu = False
data/ml10m/output/
====
data/ml10m/output/
====
data/ml10m/output/
====
data/ml10m/output/
====
data/ml10m/output/
====
/content/drive/MyDrive/Colab Notebooks/DIVRS_output/ml10m-DIVRS-0.8-cause-0.2-Res-debug_2026-06-16-16-32-50/ckpt/epoch_34.pth
100% 297/297 [00:33<00:00,  8.76it/s]
100% 297/297 [00:38<00:00,  7.75it/s]
100% 297/297 [01:00<00:00,  4.88it/s]
100% 297/297 [01:27<00:00,  3.41it/s]
100% 297/297 [01:13<00:00,  4.03it/s]
 17% 50/297 [00:05<00:26,  9.15it/s]

 19% 55/297 [00:06<00:25,  9.53it/s]

: 

## 4. In metric chính thức (Recall / HR / NDCG mọi Top-K)

In [ ]:
import glob, os, re
import pandas as pd

ev = max(glob.glob('eval_out/*/'), key=os.path.getmtime)
print('Eval run:', ev)

# Doc moi file log (bo qua file loi: symlink .INFO hong tren Drive)
text = ''
for f in glob.glob(ev + 'test_log/*'):
    try:
        with open(f, encoding='utf-8', errors='ignore') as fh:
            text += fh.read() + '\n'
    except Exception:
        pass

# Parse: moi block "TEST results topk = K:" + 3 dong recall/hit_ratio/ndcg ngay sau
results = {}
for m in re.finditer(
        r'TEST results topk = (\d+):.*?'
        r'recall:\s*([\d.]+).*?'
        r'hit_ratio:\s*([\d.]+).*?'
        r'ndcg:\s*([\d.]+)',
        text, re.DOTALL):
    K = int(m.group(1))
    results[K] = (float(m.group(2)), float(m.group(3)), float(m.group(4)))

if not results:
    print('Chua co ket qua — chay cell 3 (TEST) truoc')
else:
    df = pd.DataFrame(
        [(K, r, hr, n) for K, (r, hr, n) in sorted(results.items())],
        columns=['Top-K', 'Recall', 'HR', 'NDCG'],
    ).round(4)
    print()
    print(df.to_string(index=False))

    # Doi chieu paper DIVRS (MF, ML-10M) @20 va @50
    paper = {
        20: {'Recall': 0.1691, 'HR': 0.5302, 'NDCG': 0.1128},
        50: {'Recall': 0.2956, 'HR': 0.7041, 'NDCG': 0.1521},
    }
    for K in (20, 50):
        if K in results and K in paper:
            r, hr, n = results[K]
            mine = {'Recall': r, 'HR': hr, 'NDCG': n}
            print('\n=== Doi chieu paper @%d ===' % K)
            cmp = pd.DataFrame(
                [(metric, paper[K][metric], round(mine[metric], 4),
                  round(mine[metric] / paper[K][metric] * 100, 1))
                 for metric in ('Recall', 'HR', 'NDCG')],
                columns=['metric', 'paper', 'minh', '% khop'],
            )
            print(cmp.to_string(index=False))


Eval run: eval_out/ml10m-DIVRS-0.8-cause-0.2-Res-debug_2026-06-17-15-33-48/

 Top-K  Recall     HR   NDCG
     5  0.0636 0.2615 0.0801
    10  0.1047 0.3865 0.0913
    20  0.1675 0.5277 0.1118
    30  0.2157 0.6088 0.1275
    40  0.2559 0.6608 0.1400
    50  0.2911 0.6993 0.1505

=== Doi chieu paper @20 ===
metric  paper   minh  % khop
Recall 0.1691 0.1675    99.1
    HR 0.5302 0.5277    99.5
  NDCG 0.1128 0.1118    99.1

=== Doi chieu paper @50 ===
metric  paper   minh  % khop
Recall 0.2956 0.2911    98.5
    HR 0.7041 0.6993    99.3
  NDCG 0.1521 0.1505    98.9


## Ghi chú
- Đây là số **đúng protocol code/paper** — dùng cho báo cáo.
- Đối chiếu paper DIVRS (MF, ML-10M): **Recall@20=0.1691, HR@20=0.5302, NDCG@20=0.1128**.
- Nếu số ở đây gần các giá trị đó → tái lập thành công. Thấp hơn → cần train thêm epoch.